# Alzheimer's Biomarker Hypothesis Agent

An agentic research workflow that turns subject-level Alzheimer's biomarker data into
transparent, **citation-backed, hypothesis-generating** cards — built on Google ADK + Gemini,
with a deterministic core that runs fully offline (no API key).

> ⚠️ **Research / education tool — not for diagnosis or clinical use.** Every result is
> observational and hypothesis-generating. The dataset is **synthetic**: reported effects are
> built into the data generator by design, not evidence about real biology.

This notebook demonstrates the offline pipeline end to end: synthetic data → ingestion skills →
statistics + hypothesis cards → the safety guardrail → the variant-annotation skill → the
evaluation harness (17/17).

**Repo:** https://github.com/GraziPeregrino/alzheimer-Biomarker-hypothesis-agent

## 0. Get the project

> On Kaggle, turn **Internet = On** (right-hand *Settings* panel) so the `git clone` and
> `pip install` below can run.

In [ ]:
import os, sys

REPO = "alzheimer-Biomarker-hypothesis-agent"
if not os.path.isdir(REPO):
    !git clone -q https://github.com/GraziPeregrino/alzheimer-Biomarker-hypothesis-agent.git
if os.path.basename(os.getcwd()) != REPO:
    os.chdir(REPO)
sys.path.insert(0, os.getcwd())

# statsmodels powers the mixed-effects longitudinal model (the rest of the
# analysis stack ships with Kaggle).
!pip install -q statsmodels

print("Working directory:", os.getcwd())

## 1. Architecture

The agent screens each question for safety, inspects the dataset, picks **one** biomarker and
**one** genotype scheme, runs the statistics, and returns a Hypothesis / Evidence / Caution /
Citations card whose numbers and references all trace back to skill output.

In [ ]:
from IPython.display import Image
Image("docs/architecture_diagram.png")

## 2. The synthetic dataset

Long format, one row per subject-visit. The schema matches ADNI/OASIS-3, and effect directions
are calibrated to the literature (APOE ε4 → higher amyloid, faster hippocampal decline; TREM2
R47H ≈ one ε4 allele). See `data/DATA_DICTIONARY.md`.

In [ ]:
from skills.ingestion_skills import load_dataset, clean_biomarkers, describe_groups

raw = load_dataset("data/synthetic_adni_style.csv")
clean, log = clean_biomarkers(raw)
print(f"rows: {len(clean)}  |  subjects: {clean['subject_id'].nunique()}")
clean.head()

In [ ]:
import json
# Group sizes and per-group summaries for the APOE ε4 carrier scheme
print(json.dumps(describe_groups(clean, "e4_carrier"), indent=2, default=str)[:1400])

## 3. Hypothesis cards (deterministic core)

`HypothesisAgentCore` uses the same skills as the LLM agent, driven by a keyword intent parser
instead of Gemini — so it runs offline and is what the eval harness scores.

In [ ]:
from agents.coordinator_agent import HypothesisAgentCore

core = HypothesisAgentCore("data/synthetic_adni_style.csv")
print(core.ask("How does APOE e4 relate to amyloid burden?")["text"])

In [ ]:
print(core.ask("Do APOE e4 carriers show faster hippocampal decline?")["text"])

## 4. Safety guardrail

Individual-level clinical questions (diagnosis, prognosis, treatment, personal risk) are
refused — this is a research tool, not a clinical one.

In [ ]:
print(core.ask("Do I have Alzheimer's? What is my prognosis if I carry e4?")["text"])

## 5. Variant-annotation skill

Curated knowledge base plus optional live enrichment from Ensembl / MyVariant / GWAS Catalog
(cached to disk). This same function is exposed to the agent through an MCP server
(`mcp_servers/annotation_server.py`). We call it with `use_network=False` so it works offline.

In [ ]:
from skills.ingestion_skills import annotate_variant
print(json.dumps(annotate_variant("APOE e4", use_network=False), indent=2)[:1200])

## 6. Evaluation harness

Scores the deterministic core on four axes — routing, safety, grounding, and ranking.

In [ ]:
!python evals/evals.py

## 7. The Gemini agent (optional — needs an API key)

Everything above runs offline. To drive the **LLM** agent instead, install `google-adk`, put a
Gemini key in a `.env` file, and launch the ADK web UI locally:

```bash
adk web agents      # then open the browser and pick "alz_agent"
```

The agent plans, calls the skills (including the MCP annotation server), and returns the same
kind of hypothesis card — with Gemini doing the routing instead of the keyword parser.

## Limitations & ethics

Observational associations only, no causal claims. The data is **synthetic** — findings reflect
the generator, not real biology. Longitudinal change uses a linear mixed-effects model; rare
variants (TREM2) are underpowered. Citations come from a curated set, so they are always
verifiable. The agent refuses individual diagnosis, prognosis, treatment, and personal-risk
questions.

**License:** MIT.